In [1]:
#Going to try balancing out dataset

In [2]:
#Went through and filtered out bad quality audio examples by hand, as well as removed instances from more dominant classes. Kept instances that more closely highlighted the correct scale

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
import tensorflow as tf
from tensorflow import keras

In [5]:
keys = ['A', 'Ab', 'B', 'Bb', 'C', 'D', 'Db', 'E', 'Eb', 'F', 'G', 'Gb']

In [6]:
from keras.preprocessing import image

def load_images_from_path(path, label):
    images = []
    labels = []

    for file in os.listdir(path):
        images.append(image.img_to_array(image.load_img(os.path.join(path, file), target_size=(224, 224, 3))))
        labels.append((label))
        
    return images, labels

def show_images(images):
    fig, axes = plt.subplots(1, 8, figsize=(20, 20), subplot_kw={'xticks': [], 'yticks': []})

    for i, ax in enumerate(axes.flat):
        ax.imshow(images[i] / 255)
        
x = []
y = []

In [ ]:
label_counter = 0

for k in keys:
    
    images, labels = load_images_from_path(f"../Data/processed/Chromagram/{k}", label_counter)
    
    x += images
    y += labels

    label_counter = label_counter + 1

In [8]:
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, stratify=y, test_size=0.2, random_state=42)

x_train_norm = np.array(x_train) / 255
x_test_norm = np.array(x_test) / 255

y_train_encoded = to_categorical(y_train)
y_test_encoded = to_categorical(y_test)

In [9]:
test_ex_X = x_train_norm[239]
x_train_norm = x_train_norm[:239]

test_ex_Y = y_train_encoded[239]
y_train_encoded = y_train_encoded[:239]

In [10]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Dropout
from keras.layers import Flatten, Dense

In [11]:
model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)))
model.add(MaxPooling2D(2, 2))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D(2, 2))
model.add(Flatten())
model.add(Dense(12, activation='softmax'))
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

C:\Users\LShel\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 128)  │        36,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 18432)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 12)             │       221,196 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 554,252 (2.11 MB)

 Trainable params: 554,252 (2.11 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
hist = model.fit(x_train_norm, y_train_encoded, validation_data=(x_test_norm, y_test_encoded), batch_size=10, epochs=10)

Epoch 1/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 11s 367ms/step - accuracy: 0.0874 - loss: 2.5258 - val_accuracy: 0.1475 - val_loss: 2.4497
Epoch 2/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 351ms/step - accuracy: 0.1191 - loss: 2.4550 - val_accuracy: 0.1475 - val_loss: 2.4225
Epoch 3/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 336ms/step - accuracy: 0.1915 - loss: 2.3426 - val_accuracy: 0.5246 - val_loss: 1.4278
Epoch 4/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 351ms/step - accuracy: 0.7194 - loss: 1.0330 - val_accuracy: 0.6721 - val_loss: 0.9827
Epoch 5/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 9s 354ms/step - accuracy: 0.8254 - loss: 0.5096 - val_accuracy: 0.7869 - val_loss: 0.7745
Epoch 6/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 343ms/step - accuracy: 0.9231 - loss: 0.2925 - val_accuracy: 0.6885 - val_loss: 0.9423
Epoch 7/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 350ms/step - accuracy: 0.9598 - loss: 0.1660 - val_accuracy: 0.7705 - val_loss: 0.8803
Epoch 8/10
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 345ms/step - accuracy: 0.9836 - loss: 0.0677 - val_accuracy: 0

In [13]:
test_ex_X_batch = np.expand_dims(test_ex_X, axis=0)

prediction = model.predict(test_ex_X_batch)

predicted_class = np.argmax(prediction)
print("Predicted class:", predicted_class)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step
Predicted class: 9


In [14]:
np.argmax(test_ex_Y)

9

In [ ]:
#Going to try an instance of different song length to see what happens, also building input pre-processing pipeline

In [3]:
import librosa
import librosa.display
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
test_path = 'C:/Users/LShel/OneDrive/Documents/Applied_Machine_Learning/Datasets/Data_2/Spectrogram/Test instances/127 Am.wav'

In [5]:
def create_chromagram(file_path):
    x, sampling_rate = librosa.load(file_path)
    S = librosa.stft(x)
    H, P = librosa.decompose.hpss(S)
    chroma = librosa.feature.chroma_stft(S=np.abs(H), sr=sampling_rate)
    return chroma

In [6]:
def chromagram_to_image(chroma, target_size=(224, 224)):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.axis('off')
    librosa.display.specshow(chroma, y_axis=None, x_axis=None, ax=ax)

    buf = BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
    plt.close(fig)
    buf.seek(0)

    img = Image.open(buf).convert('RGB') 
    img = img.resize(target_size)        

    img_array = np.array(img)             
    return img_array

In [7]:
t = create_chromagram(test_path)
i = chromagram_to_image(t)

C:\Users\LShel\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


In [8]:
i_norm = i / 255

In [9]:
i_norm_batch = np.expand_dims(i_norm, axis=0)

In [10]:
i_norm_batch.shape

(1, 224, 224, 3)

In [41]:
p = model.predict(i_norm_batch)

p_c = np.argmax(p)
print("Predicted class:", p_c)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
Predicted class: 0


In [44]:
# 0 corresponds to Aminor, which is corrrect! So looks like the model works for audio clips of variable length

In [45]:
#Going to save model now, will come back and try to make more improvements to performance

In [ ]:
model.save('../Saved_Models/best.h5')

In [47]:
keras.backend.clear_session()

In [50]:
#also consider the distrubution of classes in test and training sets